In [31]:
# Phase A: Imports

from pathlib import Path
import os
import re
import json
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

from sklearn.metrics import f1_score, average_precision_score, classification_report

from openai import OpenAI

In [32]:
# Phase B: Project paths, seed, and API loading

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

load_dotenv(PROJECT_ROOT / ".env")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env")

client = OpenAI(api_key=api_key)

print("Project root:", PROJECT_ROOT)
print("API key loaded:", "Yes" if api_key else "No")

Project root: c:\Users\farha\code\classical model vs fine tuned transformer vs LLM prompting
API key loaded: Yes


In [33]:
# Phase C: Load the held-out test split

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

test_df = pd.read_csv(DATA_DIR / "test_split.csv")

print("Test shape:", test_df.shape)
display(test_df.head())

Test shape: (23936, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,dabc734316be73e5,Arbitration Committee ruling The case against ...,0,0,0,0,0,0
1,9c65828b92251c34,Rewording a proposal to which several other ed...,0,0,0,0,0,0
2,a4bf029b82e27cbe,"If it is anything like Dana Roberts' book, it ...",0,0,0,0,0,0
3,5c9fde117406a8de,""" I'm confused here, too. So far, I've only ad...",0,0,0,0,0,0
4,729c312836e017e7,it was in an offensive tone yes. but where is ...,0,0,0,0,0,0


In [34]:
# Phase D: Create a manageable LLM evaluation subset

test_df["label_count"] = test_df[label_cols].sum(axis=1)
test_df["has_any_label"] = (test_df["label_count"] > 0).astype(int)

# Target sample size
TOTAL_SAMPLE_SIZE = 600
POS_SAMPLE_SIZE = 300
NEG_SAMPLE_SIZE = 300

positive_df = test_df[test_df["has_any_label"] == 1]
negative_df = test_df[test_df["has_any_label"] == 0]

sampled_positive = positive_df.sample(
    n=min(POS_SAMPLE_SIZE, len(positive_df)),
    random_state=SEED
)

sampled_negative = negative_df.sample(
    n=min(NEG_SAMPLE_SIZE, len(negative_df)),
    random_state=SEED
)

llm_df = pd.concat([sampled_positive, sampled_negative]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("LLM evaluation subset shape:", llm_df.shape)
print("Positive examples:", llm_df["has_any_label"].sum())
print("Negative examples:", (llm_df["has_any_label"] == 0).sum())

display(llm_df.head())

LLM evaluation subset shape: (600, 10)
Positive examples: 300
Negative examples: 300


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,label_count,has_any_label
0,c499a1bfde09f787,"BEWARE OF FLYING DICKS, FUDGING NIGGER.",1,0,1,0,1,0,3,1
1,48e1bbd97ad0b109,WOW I had no clue that Lori quit plaing drums ...,0,0,0,0,0,0,0,0
2,22d6ba7ed2591367,Archived copy (talk • contributions • email),0,0,0,0,0,0,0,0
3,2cf09da86916d420,""" WHO IS ATTACKING WHO? WHATEVER, THE FACT THA...",1,0,1,0,0,0,2,1
4,ed8c1e61dc20e65f,"STAY OFF WIKI YOU KIKE, YOU ARE INHERENTLY BIA...",1,0,0,0,0,0,1,1


In [35]:
# Phase E: Prompt construction and label parsing setup

valid_labels = set(label_cols)

def build_zero_shot_prompt(comment_text: str) -> str:
    return f"""
You are a multi-label toxic comment classifier.

Possible labels:
- toxic
- severe_toxic
- obscene
- threat
- insult
- identity_hate

Task:
Read the comment and return all applicable labels as a JSON array.
If none apply, return an empty JSON array [].

Rules:
- Only use labels from the allowed list above.
- Return JSON only.
- Do not explain your answer.
- Multiple labels are allowed.

Comment:
\"\"\"{comment_text}\"\"\"
""".strip()


def parse_labels_from_response(text: str):
    """
    Try to parse a JSON list of labels from the model response.
    Return a sorted list of valid labels only.
    """
    text = text.strip()

    # Try direct JSON parse first
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            cleaned = sorted([label for label in parsed if label in valid_labels])
            return cleaned
    except:
        pass

    # Fallback: find labels in raw text
    found = []
    for label in label_cols:
        if re.search(rf"\b{re.escape(label)}\b", text):
            found.append(label)

    return sorted(set(found))

In [36]:
# Phase F: Sanity check one prompt

example_text = llm_df.loc[0, "comment_text"]
print(build_zero_shot_prompt(example_text))

You are a multi-label toxic comment classifier.

Possible labels:
- toxic
- severe_toxic
- obscene
- threat
- insult
- identity_hate

Task:
Read the comment and return all applicable labels as a JSON array.
If none apply, return an empty JSON array [].

Rules:
- Only use labels from the allowed list above.
- Return JSON only.
- Do not explain your answer.
- Multiple labels are allowed.

Comment:
"""BEWARE OF FLYING DICKS, FUDGING NIGGER."""


In [37]:
# Phase G: OpenAI call helper

MODEL_NAME = "gpt-4.1-mini"  # good balance of cost and quality for classification-style prompting

def get_llm_prediction(comment_text: str, model_name: str = MODEL_NAME):
    prompt = build_zero_shot_prompt(comment_text)

    response = client.responses.create(
        model=model_name,
        input=prompt,
        temperature=0
    )

    raw_text = response.output_text.strip()
    parsed_labels = parse_labels_from_response(raw_text)

    return raw_text, parsed_labels

In [38]:
# Phase H: Test one example before running the full subset

test_comment = llm_df.loc[0, "comment_text"]
raw_output, parsed_output = get_llm_prediction(test_comment)

print("RAW OUTPUT:")
print(raw_output)
print("\nPARSED LABELS:")
print(parsed_output)

RAW OUTPUT:
["toxic","severe_toxic","obscene","insult","identity_hate"]

PARSED LABELS:
['identity_hate', 'insult', 'obscene', 'severe_toxic', 'toxic']


In [39]:
# Phase I: Run zero-shot prompting on the sampled subset

raw_outputs = []
parsed_outputs = []

for _, row in tqdm(llm_df.iterrows(), total=len(llm_df), desc="LLM inference"):
    comment_text = row["comment_text"]
    raw_text, parsed_labels = get_llm_prediction(comment_text)

    raw_outputs.append(raw_text)
    parsed_outputs.append(parsed_labels)

llm_df["raw_output"] = raw_outputs
llm_df["parsed_labels"] = parsed_outputs

print("LLM inference complete.")
display(llm_df[["id", "comment_text", "raw_output", "parsed_labels"]].head())

LLM inference:   0%|          | 0/600 [00:00<?, ?it/s]

LLM inference complete.


,id,comment_text,raw_output,parsed_labels
0,c499a1bfde09f787,"BEWARE OF FLYING DICKS, FUDGING NIGGER.","[""toxic"",""severe_toxic"",""obscene"",""insult"",""id...","[identity_hate, insult, obscene, severe_toxic,..."
1,48e1bbd97ad0b109,WOW I had no clue that Lori quit plaing drums ...,[],[]
2,22d6ba7ed2591367,Archived copy (talk • contributions • email),[],[]
3,2cf09da86916d420,""" WHO IS ATTACKING WHO? WHATEVER, THE FACT THA...","[""toxic"",""obscene"",""insult""]","[insult, obscene, toxic]"
4,ed8c1e61dc20e65f,"STAY OFF WIKI YOU KIKE, YOU ARE INHERENTLY BIA...","[""toxic"",""severe_toxic"",""obscene"",""insult"",""id...","[identity_hate, insult, obscene, severe_toxic,..."


In [40]:
# Phase J: Convert parsed label lists into binary prediction columns

for label in label_cols:
    llm_df[f"{label}_pred"] = llm_df["parsed_labels"].apply(lambda labels: 1 if label in labels else 0)

print("Prediction columns created.")
display(llm_df[[f"{label}_pred" for label in label_cols]].head())

Prediction columns created.


,toxic_pred,severe_toxic_pred,obscene_pred,threat_pred,insult_pred,identity_hate_pred
0,1,1,1,0,1,1
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,1,0,1,0,1,0
4,1,1,1,0,1,1


In [41]:
# Phase K: Build y_true and y_pred matrices

y_true = llm_df[label_cols].values
y_pred = llm_df[[f"{label}_pred" for label in label_cols]].values

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)

y_true shape: (600, 6)
y_pred shape: (600, 6)


In [42]:
# Phase L: Compute overall LLM metrics

micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

overall_metrics = pd.DataFrame({
    "metric": ["micro_f1", "macro_f1"],
    "value": [micro_f1, macro_f1]
})

display(overall_metrics)

,metric,value
0,micro_f1,0.743290
1,macro_f1,0.578334


In [43]:
# Phase M: Compute per-label F1

per_label_rows = []

for i, label in enumerate(label_cols):
    label_f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
    per_label_rows.append({
        "label": label,
        "f1": label_f1
    })

per_label_metrics = pd.DataFrame(per_label_rows)
display(per_label_metrics)

,label,f1
0,toxic,0.880866
1,severe_toxic,0.130435
2,obscene,0.801262
3,threat,0.565217
4,insult,0.670000
5,identity_hate,0.422222


In [44]:
# Phase N: Save LLM results

subset_path = RESULTS_DIR / "llm_eval_subset_predictions.csv"
overall_metrics_path = RESULTS_DIR / "llm_zero_shot_overall_metrics.csv"
per_label_metrics_path = RESULTS_DIR / "llm_zero_shot_per_label_metrics.csv"

llm_df.to_csv(subset_path, index=False)
overall_metrics.to_csv(overall_metrics_path, index=False)
per_label_metrics.to_csv(per_label_metrics_path, index=False)

print("Saved:")
print(subset_path)
print(overall_metrics_path)
print(per_label_metrics_path)

Saved:
c:\Users\farha\code\classical model vs fine tuned transformer vs LLM prompting\results\llm_eval_subset_predictions.csv
c:\Users\farha\code\classical model vs fine tuned transformer vs LLM prompting\results\llm_zero_shot_overall_metrics.csv
c:\Users\farha\code\classical model vs fine tuned transformer vs LLM prompting\results\llm_zero_shot_per_label_metrics.csv


In [45]:
# Phase O: Print classification report for each label

for i, label in enumerate(label_cols):
    print(f"\n===== {label} =====")
    print(classification_report(
        y_true[:, i],
        y_pred[:, i],
        zero_division=0
    ))


===== toxic =====
              precision    recall  f1-score   support

           0       0.87      0.92      0.90       314
           1       0.91      0.85      0.88       286

    accuracy                           0.89       600
   macro avg       0.89      0.89      0.89       600
weighted avg       0.89      0.89      0.89       600


===== severe_toxic =====
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       576
           1       0.14      0.12      0.13        24

    accuracy                           0.93       600
   macro avg       0.55      0.55      0.55       600
weighted avg       0.93      0.93      0.93       600


===== obscene =====
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       440
           1       0.81      0.79      0.80       160

    accuracy                           0.90       600
   macro avg       0.87      0.86      0.86       600
weighted

Few Shot

In [46]:
# Phase P: Load training split for few-shot demonstration retrieval

train_df = pd.read_csv(DATA_DIR / "train_split.csv")

print("Train split shape:", train_df.shape)
display(train_df.head())

Train split shape: (111699, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,d52a83440fdc89a7,""" Islands in the Caribbean Wondering why you r...",0,0,0,0,0,0
1,4e1e119b380a5d9e,"""add the complaint, however please read the in...",0,0,0,0,0,0
2,1bfd3b5c49f96bd6,"""I suggest that there be a """"Lincoln High Scho...",0,0,0,0,0,0
3,ebaadb1e33cc8f14,oi I've had enough of your threats mr. Sort yo...,0,0,0,0,0,0
4,27475b04e576e998,Article To Long ?! I am wondering if the Dog a...,0,0,0,0,0,0


In [47]:
# Phase Q: Helper to convert row labels into JSON-style label lists

def row_to_label_list(row, label_cols):
    labels = [label for label in label_cols if int(row[label]) == 1]
    return labels

In [48]:
# Phase R: TF-IDF retrieval setup for few-shot prompting

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

retrieval_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=20000,
    min_df=2,
    sublinear_tf=True
)

train_texts = train_df["comment_text"].fillna("").astype(str).tolist()
train_tfidf = retrieval_vectorizer.fit_transform(train_texts)

print("Train retrieval TF-IDF shape:", train_tfidf.shape)

Train retrieval TF-IDF shape: (111699, 20000)


In [49]:
# Phase S: Retrieve nearest training examples for few-shot prompting

FEW_SHOT_K = 4

def retrieve_few_shot_examples(comment_text, k=FEW_SHOT_K):
    query_vec = retrieval_vectorizer.transform([comment_text])
    sims = cosine_similarity(query_vec, train_tfidf).flatten()

    top_indices = np.argsort(sims)[::-1]

    examples = []
    seen_texts = set()

    for idx in top_indices:
        ex_text = str(train_df.iloc[idx]["comment_text"]).strip()

        if ex_text == comment_text.strip():
            continue
        if ex_text in seen_texts:
            continue

        seen_texts.add(ex_text)

        ex_labels = row_to_label_list(train_df.iloc[idx], label_cols)
        examples.append({
            "comment_text": ex_text,
            "labels": ex_labels
        })

        if len(examples) == k:
            break

    return examples

In [50]:
# Phase T: Few-shot prompt builder

def build_few_shot_prompt(comment_text: str, examples: list) -> str:
    demo_blocks = []

    for ex in examples:
        demo_blocks.append(
            f'Comment:\n"""{ex["comment_text"]}"""\nLabels:\n{json.dumps(ex["labels"])}'
        )

    demo_text = "\n\n".join(demo_blocks)

    prompt = f"""
You are a multi-label toxic comment classifier.

Possible labels:
- toxic
- severe_toxic
- obscene
- threat
- insult
- identity_hate

Task:
Read the comment and return all applicable labels as a JSON array.
If none apply, return an empty JSON array [].

Rules:
- Only use labels from the allowed list above.
- Return JSON only.
- Do not explain your answer.
- Multiple labels are allowed.

Examples:
{demo_text}

Now classify the following comment.

Comment:
\"\"\"{comment_text}\"\"\"
""".strip()

    return prompt

In [51]:
# Phase U: Few-shot OpenAI call helper

def get_llm_prediction_few_shot(comment_text: str, model_name: str = MODEL_NAME):
    examples = retrieve_few_shot_examples(comment_text, k=FEW_SHOT_K)
    prompt = build_few_shot_prompt(comment_text, examples)

    response = client.responses.create(
        model=model_name,
        input=prompt,
        temperature=0
    )

    raw_text = response.output_text.strip()
    parsed_labels = parse_labels_from_response(raw_text)

    return raw_text, parsed_labels, examples

In [52]:
# Phase V: Sanity check one few-shot example

test_comment = llm_df.loc[0, "comment_text"]
retrieved_examples = retrieve_few_shot_examples(test_comment, k=FEW_SHOT_K)

print(build_few_shot_prompt(test_comment, retrieved_examples)[:4000])  # print first part only

You are a multi-label toxic comment classifier.

Possible labels:
- toxic
- severe_toxic
- obscene
- threat
- insult
- identity_hate

Task:
Read the comment and return all applicable labels as a JSON array.
If none apply, return an empty JSON array [].

Rules:
- Only use labels from the allowed list above.
- Return JSON only.
- Do not explain your answer.
- Multiple labels are allowed.

Examples:
Comment:
"""your a faggot and a nigger"""
Labels:
["toxic", "severe_toxic", "obscene", "insult", "identity_hate"]

Comment:
"""" fisto is a retard nigger.""""
Labels:
["toxic", "obscene", "insult", "identity_hate"]

Comment:
"""Oh Fuck you, nigger."""
Labels:
["toxic", "obscene", "insult", "identity_hate"]

Comment:
"""I am a fucking retarded nigger"""
Labels:
["toxic", "obscene", "insult", "identity_hate"]

Now classify the following comment.

Comment:
"""BEWARE OF FLYING DICKS, FUDGING NIGGER."""


In [53]:
# Phase W: Test one few-shot prediction

raw_output_fs, parsed_output_fs, used_examples = get_llm_prediction_few_shot(test_comment)

print("RAW OUTPUT:")
print(raw_output_fs)

print("\nPARSED LABELS:")
print(parsed_output_fs)

print("\nFIRST RETRIEVED EXAMPLE LABELS:")
print(used_examples[0]["labels"] if used_examples else "No examples retrieved")

RAW OUTPUT:
["toxic", "severe_toxic", "obscene", "insult", "identity_hate"]

PARSED LABELS:
['identity_hate', 'insult', 'obscene', 'severe_toxic', 'toxic']

FIRST RETRIEVED EXAMPLE LABELS:
['toxic', 'severe_toxic', 'obscene', 'insult', 'identity_hate']


In [54]:
# Phase X: Run few-shot prompting on the full sampled subset

few_shot_raw_outputs = []
few_shot_parsed_outputs = []
few_shot_examples_used = []

for _, row in tqdm(llm_df.iterrows(), total=len(llm_df), desc="Few-shot LLM inference"):
    comment_text = row["comment_text"]
    raw_text, parsed_labels, examples = get_llm_prediction_few_shot(comment_text)

    few_shot_raw_outputs.append(raw_text)
    few_shot_parsed_outputs.append(parsed_labels)
    few_shot_examples_used.append(examples)

llm_df["few_shot_raw_output"] = few_shot_raw_outputs
llm_df["few_shot_parsed_labels"] = few_shot_parsed_outputs
llm_df["few_shot_examples_used"] = few_shot_examples_used

print("Few-shot inference complete.")
display(llm_df[["id", "comment_text", "few_shot_raw_output", "few_shot_parsed_labels"]].head())

Few-shot LLM inference:   0%|          | 0/600 [00:00<?, ?it/s]

Few-shot inference complete.


,id,comment_text,few_shot_raw_output,few_shot_parsed_labels
0,c499a1bfde09f787,"BEWARE OF FLYING DICKS, FUDGING NIGGER.","[""toxic"", ""severe_toxic"", ""obscene"", ""insult"",...","[identity_hate, insult, obscene, severe_toxic,..."
1,48e1bbd97ad0b109,WOW I had no clue that Lori quit plaing drums ...,[],[]
2,22d6ba7ed2591367,Archived copy (talk • contributions • email),[],[]
3,2cf09da86916d420,""" WHO IS ATTACKING WHO? WHATEVER, THE FACT THA...","[""toxic"", ""obscene"", ""insult""]","[insult, obscene, toxic]"
4,ed8c1e61dc20e65f,"STAY OFF WIKI YOU KIKE, YOU ARE INHERENTLY BIA...","[""toxic"", ""insult"", ""identity_hate""]","[identity_hate, insult, toxic]"


In [55]:
# Phase Y: Convert few-shot parsed labels into binary columns

for label in label_cols:
    llm_df[f"{label}_few_shot_pred"] = llm_df["few_shot_parsed_labels"].apply(
        lambda labels: 1 if label in labels else 0
    )

few_y_pred = llm_df[[f"{label}_few_shot_pred" for label in label_cols]].values

print("Few-shot prediction matrix shape:", few_y_pred.shape)

Few-shot prediction matrix shape: (600, 6)


In [56]:
# Phase Z: Compute few-shot overall metrics

few_micro_f1 = f1_score(y_true, few_y_pred, average="micro", zero_division=0)
few_macro_f1 = f1_score(y_true, few_y_pred, average="macro", zero_division=0)

few_overall_metrics = pd.DataFrame({
    "metric": ["micro_f1", "macro_f1"],
    "value": [few_micro_f1, few_macro_f1]
})

display(few_overall_metrics)

,metric,value
0,micro_f1,0.751743
1,macro_f1,0.611014


In [57]:
# Phase AA: Compute few-shot per-label F1

few_per_label_rows = []

for i, label in enumerate(label_cols):
    label_f1 = f1_score(y_true[:, i], few_y_pred[:, i], zero_division=0)
    few_per_label_rows.append({
        "label": label,
        "f1": label_f1
    })

few_per_label_metrics = pd.DataFrame(few_per_label_rows)
display(few_per_label_metrics)

,label,f1
0,toxic,0.881416
1,severe_toxic,0.333333
2,obscene,0.776699
3,threat,0.558140
4,insult,0.702703
5,identity_hate,0.413793


In [58]:
# Phase AB: Save few-shot outputs

few_subset_path = RESULTS_DIR / "llm_few_shot_eval_subset_predictions.csv"
few_overall_metrics_path = RESULTS_DIR / "llm_few_shot_overall_metrics.csv"
few_per_label_metrics_path = RESULTS_DIR / "llm_few_shot_per_label_metrics.csv"

llm_df.to_csv(few_subset_path, index=False)
few_overall_metrics.to_csv(few_overall_metrics_path, index=False)
few_per_label_metrics.to_csv(few_per_label_metrics_path, index=False)

print("Saved:")
print(few_subset_path)
print(few_overall_metrics_path)
print(few_per_label_metrics_path)

Saved:
c:\Users\farha\code\classical model vs fine tuned transformer vs LLM prompting\results\llm_few_shot_eval_subset_predictions.csv
c:\Users\farha\code\classical model vs fine tuned transformer vs LLM prompting\results\llm_few_shot_overall_metrics.csv
c:\Users\farha\code\classical model vs fine tuned transformer vs LLM prompting\results\llm_few_shot_per_label_metrics.csv


In [59]:
# Phase AC: Zero-shot vs few-shot comparison table

llm_comparison_df = pd.DataFrame({
    "llm_setting": ["Zero-shot", "Few-shot"],
    "micro_f1": [micro_f1, few_micro_f1],
    "macro_f1": [macro_f1, few_macro_f1]
})

llm_comparison_df.to_csv(RESULTS_DIR / "llm_zero_vs_few_shot_comparison.csv", index=False)
display(llm_comparison_df)

,llm_setting,micro_f1,macro_f1
0,Zero-shot,0.743290,0.578334
1,Few-shot,0.751743,0.611014


In [66]:
final_comparison_df = pd.DataFrame({
    "model": [
        "TF-IDF + Logistic Regression",
        "RoBERTa Fine-Tuning",
        "LLM Prompting (Zero-shot)",
        "LLM Prompting (Few-shot)"
    ],
    "evaluation_set": [
        "Full held-out test split",
        "Full held-out test split",
        "600-sample test subset",
        "600-sample test subset"
    ],
    "micro_f1": [0.730514, 0.782046, 0.743290, 0.751743],
    "macro_f1": [0.604852, 0.685848, 0.578334, 0.611014],
    "mean_pr_auc": [0.634141, 0.728145, np.nan, np.nan]
})

final_comparison_df.to_csv(RESULTS_DIR / "final_model_comparison.csv", index=False)
final_comparison_df

,model,evaluation_set,micro_f1,macro_f1,mean_pr_auc
0,TF-IDF + Logistic Regression,Full held-out test split,0.730514,0.604852,0.634141
1,RoBERTa Fine-Tuning,Full held-out test split,0.782046,0.685848,0.728145
2,LLM Prompting (Zero-shot),600-sample test subset,0.743290,0.578334,NaN
3,LLM Prompting (Few-shot),600-sample test subset,0.751743,0.611014,NaN


In [65]:
final_comparison_df.to_csv(RESULTS_DIR / "final_model_comparison.csv", index=False)